# 06 — Calibration: does attention change confidence even when macro F1 does not? (reviewer Q4)

CPU only, ≈ 5 min. Reads the per-image softmax outputs (`preds.csv`, lowest-validation-loss checkpoint, test set)
saved by notebook 02 for every run. No retraining.

* **ECE** — expected calibration error of the top-label confidence, 15 equal-width bins.
* **Classwise ECE** — mean over classes of the one-vs-rest ECE (sensitive to minority classes).
* **Brier score** — mean squared error of the 4-class probability vector; **NLL** — mean negative log-likelihood.
* Attention effects on ECE / Brier / NLL with the same paired hierarchical bootstrap as notebook 03
  (stratified test-image resampling shared by all models × seed resampling), Holm-corrected within each metric.
* Reliability diagrams for the three ResNet50 variants (pooled over seeds).

Post-hoc temperature scaling is not applied: the only validation data are 38 images (6 duplicated in training),
too few to fit a temperature reliably; this is stated in the paper.

In [ ]:
import os, glob, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

RESULT_GLOBS = ['/kaggle/input/**/results/*/metrics.json', '/kaggle/working/results/*/metrics.json']
OUT = '/kaggle/working/calibration'; os.makedirs(OUT, exist_ok=True)
N_BOOT, N_BINS, CKPT = 2000, 15, 'best'
RNG = np.random.default_rng(2026)
CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']; K = 4
ARCH = ['resnet50', 'densenet121', 'mobilenetv2', 'customcnn']
ARCH_LABEL = {'resnet50': 'ResNet50', 'densenet121': 'DenseNet121', 'mobilenetv2': 'MobileNetV2', 'customcnn': 'Custom CNN'}
ATT_LABEL = {'none': 'Baseline', 'se': 'SE', 'cbam': 'CBAM'}

In [ ]:
metas = {}
for g in RESULT_GLOBS:
    for f in glob.glob(g, recursive=True):
        m = json.load(open(f))
        if m.get('status') == 'done' and not m.get('smoke_test'):
            m['dir'] = os.path.dirname(f); metas.setdefault(m['run_id'], m)
RUNS, ref = {}, None
for rid, m in metas.items():
    P = pd.read_csv(os.path.join(m['dir'], 'preds.csv'))
    d = P[(P.split == 'test') & (P.ckpt == CKPT)].sort_values('idx')
    if ref is None: ref = d.relpath.tolist(); Y = d.y_true.values
    assert d.relpath.tolist() == ref, rid
    variant = ''.join(f'_{k}-{v}' for k, v in sorted(m.get('opts', {}).items()))
    RUNS[rid] = dict(cfg=(m['arch'], m['attention'], m['condition'], variant), seed=m['seed'],
                     prob=np.clip(d[[f'p_{c}' for c in CLASS_NAMES]].values, 1e-12, 1))
CFG = {}
for rid, r in RUNS.items(): CFG.setdefault(r['cfg'], {})[r['seed']] = rid
print(len(RUNS), 'runs,', len(CFG), 'configurations')

In [ ]:
# ── metrics, vectorised over bootstrap replicates ─────────────────────────────
strata = [np.where(Y == c)[0] for c in range(K)]
BOOT = np.concatenate([s[RNG.integers(0, len(s), (N_BOOT, len(s)))] for s in strata], axis=1)   # (B, n), shared
ALL = np.arange(len(Y))[None, :]

def ece_rows(conf, corr, idx, nb=N_BINS):
    """ECE for each row of idx (resampled image indices)."""
    b = np.minimum((conf * nb).astype(int), nb - 1)
    R = idx.shape[0]; off = (np.arange(R) * nb)[:, None]; key = (b[idx] + off).ravel()
    sc = np.bincount(key, weights=conf[idx].ravel(), minlength=R * nb).reshape(R, nb)
    sk = np.bincount(key, weights=corr[idx].ravel(), minlength=R * nb).reshape(R, nb)
    return np.abs(sc - sk).sum(1) / idx.shape[1]

def run_metrics(prob, idx):
    pred = prob.argmax(1); conf = prob.max(1); corr = (pred == Y).astype(float)
    onehot = np.eye(K)[Y]
    out = {'ece': ece_rows(conf, corr, idx),
           'cw_ece': np.mean([ece_rows(prob[:, c], (Y == c).astype(float), idx) for c in range(K)], 0),
           'brier': ((prob - onehot) ** 2).sum(1)[idx].mean(1),
           'nll': (-np.log(prob[np.arange(len(Y)), Y]))[idx].mean(1),
           'acc': corr[idx].mean(1)}
    return out

POINT = {rid: {k: v[0] for k, v in run_metrics(r['prob'], ALL).items()} for rid, r in RUNS.items()}
BOOTM = {rid: run_metrics(r['prob'], BOOT) for rid, r in RUNS.items()}
METRICS = ['ece', 'cw_ece', 'brier', 'nll']

def seeds(c): return sorted(CFG.get(c, {}))
def summ(c, k):
    v = np.array([POINT[CFG[c][s]][k] for s in seeds(c)]); return v.mean(), v.std(ddof=1) if len(v) > 1 else np.nan

def compare(ca, cb, k):
    common = sorted(set(seeds(ca)) & set(seeds(cb)))
    if not common: return None
    D = np.stack([BOOTM[CFG[ca][s]][k] - BOOTM[CFG[cb][s]][k] for s in common])
    pick = RNG.integers(0, len(common), (N_BOOT, len(common)))
    d = D[pick, np.arange(N_BOOT)[:, None]].mean(1)
    pv = np.array([POINT[CFG[ca][s]][k] - POINT[CFG[cb][s]][k] for s in common])
    return dict(delta=pv.mean(), ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5), n_seeds=len(common),
                seeds_positive=int((pv > 0).sum()), p_boot=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for k, i in enumerate(o): run = max(run, (m - k) * p[i]); adj[i] = min(1, run)
    return adj

In [ ]:
# ── table: calibration of all 24 main configurations ──────────────────────────
rows = []
for cond in ['imbalanced', 'balanced']:
    for a in ARCH:
        for t in ['none', 'se', 'cbam']:
            c = (a, t, cond, '')
            if c not in CFG: continue
            r = dict(condition=cond, Model=ARCH_LABEL[a], Attention=ATT_LABEL[t], n=len(seeds(c)))
            for k in METRICS:
                m, sd = summ(c, k); r[k] = f'{m:.4f} ± {sd:.4f}'; r[f'_{k}'] = m
            rows.append(r)
CAL = pd.DataFrame(rows); CAL.to_csv(f'{OUT}/calibration_table.csv', index=False)
display(CAL[[c for c in CAL.columns if not c.startswith('_')]])

In [ ]:
# ── attention effects on calibration (attention − baseline; negative = better calibrated) ──
E = []
for cond in ['imbalanced', 'balanced']:
    for a in ARCH:
        for t in ['se', 'cbam']:
            ca, cb = (a, t, cond, ''), (a, 'none', cond, '')
            if ca in CFG and cb in CFG:
                for k in METRICS:
                    r = compare(ca, cb, k)
                    if r: E.append(dict(metric=k, effect=f'{ATT_LABEL[t]} − Baseline | {ARCH_LABEL[a]} | {cond}', **r))
# within-variant effects (e.g. lung crop, grouped validation, CXR pretraining)
for c in CFG:
    a, t, cond, v = c
    if v and t != 'none':
        base = (a, 'none', cond, v) if (a, 'none', cond, v) in CFG else (a, 'none', cond, '')   # multi-depth: main baseline
        if base in CFG and any(f'_{k}-' in v for k in ('lungcrop', 'val', 'placement', 'pretrain')):
            for k in METRICS:
                r = compare(c, base, k)
                if r: E.append(dict(metric=k, effect=f'{ATT_LABEL[t]} − Baseline | {ARCH_LABEL[a]} | {cond}{v}'
                                    + ('' if base[3] else ' (vs main baseline)'), **r))
EF = pd.DataFrame(E)
if len(EF):
    EF['p_holm'] = np.nan
    for k, idx in EF.groupby('metric').groups.items(): EF.loc[idx, 'p_holm'] = holm(EF.loc[idx, 'p_boot'])
    EF['CI_excludes_0'] = (EF.ci_lo > 0) | (EF.ci_hi < 0)
    EF.to_csv(f'{OUT}/calibration_attention_effects.csv', index=False)
    with pd.option_context('display.max_colwidth', 80, 'display.width', 220):
        display(EF[EF.metric == 'ece'].round(4))
    print('effects with CI excluding 0, by metric:', EF.groupby('metric').CI_excludes_0.sum().to_dict())
    print('effects surviving Holm (p_holm < 0.05), by metric:', EF.assign(s=EF.p_holm < 0.05).groupby('metric').s.sum().to_dict())

In [ ]:
# ── reliability diagrams: ResNet50 variants, pooled over seeds ────────────────
fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
for j, cond in enumerate(['imbalanced', 'balanced']):
    ax[j].plot([0, 1], [0, 1], 'k:', lw=1)
    for t, col in zip(['none', 'se', 'cbam'], ['#4c72b0', '#dd8452', '#55a868']):
        c = ('resnet50', t, cond, '')
        if c not in CFG: continue
        conf = np.concatenate([RUNS[CFG[c][s]]['prob'].max(1) for s in seeds(c)])
        corr = np.concatenate([(RUNS[CFG[c][s]]['prob'].argmax(1) == Y) for s in seeds(c)]).astype(float)
        b = np.minimum((conf * N_BINS).astype(int), N_BINS - 1); xs, ys = [], []
        for k in range(N_BINS):
            if (b == k).sum() >= 10: xs.append(conf[b == k].mean()); ys.append(corr[b == k].mean())
        m, _ = summ(c, 'ece')
        ax[j].plot(xs, ys, 'o-', color=col, ms=4, label=f'{ATT_LABEL[t]} (ECE {m:.3f})')
    ax[j].set(xlim=(0.3, 1.0), ylim=(0.3, 1.0), xlabel='confidence (top-label)', ylabel='accuracy',
              title=f'ResNet50 — {cond} training'); ax[j].legend(fontsize=8, loc='upper left')
plt.tight_layout(); plt.savefig(f'{OUT}/reliability_resnet50.png', dpi=200); plt.show()